# Vision image smoke test

Exercises every library this image exists for, headless, with no network access and
no GPU. CI runs this on every PR. It also doubles as a worked reference for the
three things the image is actually for: sampling a point cloud from a mesh,
registering two point clouds, and recovering camera poses from a set of images.

In [ ]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2, skimage, imageio, trimesh, open3d as o3d, pycolmap, torch

for name, mod in [("opencv", cv2), ("scikit-image", skimage), ("imageio", imageio),
                  ("trimesh", trimesh), ("open3d", o3d), ("pycolmap", pycolmap),
                  ("torch", torch), ("numpy", np)]:
    print(f"{name:14s} {mod.__version__}")
print("pycolmap built with CUDA:", pycolmap.has_cuda)
print("torch sees a GPU:", torch.cuda.is_available())

## Open3D: sample a point cloud from a mesh

This is the activity the image was requested for. The mesh is built locally rather
than downloaded so the test does not depend on the network.

In [ ]:
mesh = o3d.geometry.TriangleMesh.create_torus(torus_radius=1.0, tube_radius=0.35,
                                              radial_resolution=60, tubular_resolution=30)
mesh.compute_vertex_normals()
print(f"mesh: {len(mesh.vertices)} vertices, {len(mesh.triangles)} triangles")

pcd = mesh.sample_points_poisson_disk(number_of_points=4000)
pts = np.asarray(pcd.points)
print(f"sampled {len(pts)} points, bbox {np.round(pts.min(0),2)} .. {np.round(pts.max(0),2)}")
assert len(pts) == 4000

# visualised with matplotlib, which is all the course needs - no OpenGL involved
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(projection="3d")
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=1, c=pts[:, 2], cmap="viridis")
ax.set_box_aspect((1, 1, 0.5)); ax.set_title("point cloud sampled from a mesh")
plt.close(fig)
print("matplotlib rendered the cloud without a display")

## Open3D: register two point clouds with ICP

In [ ]:
rng = np.random.default_rng(0)
src = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts))
T = np.eye(4); T[:3, 3] = [0.05, -0.03, 0.04]
tgt = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts)).transform(T)

reg = o3d.pipelines.registration.registration_icp(
    src, tgt, 0.3, np.eye(4),
    o3d.pipelines.registration.TransformationEstimationPointToPoint())
recovered = reg.transformation[:3, 3]
print(f"fitness {reg.fitness:.4f}  rmse {reg.inlier_rmse:.5f}")
print(f"recovered translation {np.round(recovered, 4)}  (true {T[:3, 3]})")
assert reg.fitness > 0.9
assert np.allclose(recovered, T[:3, 3], atol=5e-3)

down = src.voxel_down_sample(0.1)
print(f"voxel downsample: {len(src.points)} -> {len(down.points)} points")

## trimesh: mesh properties, ray casting, format round-trip

In [ ]:
import io
m = trimesh.creation.icosphere(subdivisions=3, radius=1.0)
print(f"watertight={m.is_watertight}  volume={m.volume:.4f}  (4/3 pi = {4/3*np.pi:.4f})")
assert m.is_watertight and abs(m.volume - 4/3*np.pi) < 0.05

hits = m.ray.intersects_location(ray_origins=np.array([[0., 0., -5.]]),
                                 ray_directions=np.array([[0., 0., 1.]]))[0]
print(f"ray through the centre crosses {len(hits)} surfaces at z={np.round(sorted(hits[:,2]),3)}")
assert len(hits) == 2

blob = m.export(file_type="ply")
m2 = trimesh.load(io.BytesIO(blob), file_type="ply")
assert len(m2.vertices) == len(m.vertices)
print(f"ply round-trip preserved {len(m2.vertices)} vertices")

# trimesh and open3d hand meshes to each other through plain numpy arrays
o3d_mesh = o3d.geometry.TriangleMesh(o3d.utility.Vector3dVector(np.asarray(m.vertices)),
                                     o3d.utility.Vector3iVector(np.asarray(m.faces)))
o3d_mesh.compute_vertex_normals()
print(f"handed to open3d: watertight={o3d_mesh.is_watertight()}")

## A synthetic scene for structure from motion

Three non-coplanar textured planes, viewed from a short camera arc. Non-coplanar
matters: a single plane makes the SfM problem degenerate and COLMAP will refuse it.

In [ ]:
import os, tempfile
W, H, F = 1024, 768, 900.0
K = np.array([[F, 0, W/2], [0, F, H/2], [0, 0, 1]])

def texture(seed, size=900):
    r = np.random.default_rng(seed)
    coarse = cv2.resize(r.random((size//6, size//6, 3)), (size, size), interpolation=cv2.INTER_CUBIC)
    coarse = (coarse - coarse.min()) / (np.ptp(coarse) + 1e-9)
    return (np.clip(0.75*coarse + 0.25*r.random((size, size, 1)), 0, 1) * 255).astype(np.uint8)

PLANES = [np.float32([[-1,-1,4],[1,-1,4],[1,1,4],[-1,1,4]]),
          np.float32([[-1,1,4],[1,1,4],[1,1.2,2],[-1,1.2,2]]),
          np.float32([[-1,-1,4],[-1,1,4],[-1.2,1,2],[-1.2,-1,2]])]
TEX = [texture(s) for s in (1, 2, 3)]

def render(i, n):
    a = -0.55 + 1.1 * (i / max(n-1, 1))
    eye = np.array([2.6*np.sin(a), -0.15 + 0.35*np.sin(3*a), 4 - 2.6*np.cos(a)])
    f = np.array([0., 0., 3.6]) - eye; f /= np.linalg.norm(f)
    r = np.cross(np.array([0., -1, 0]), f); r /= np.linalg.norm(r)
    R = np.stack([r, np.cross(f, r), f]); t = -R @ eye
    img = np.zeros((H, W, 3), np.uint8)
    for corners, tex in zip(PLANES, TEX):
        cam = (R @ corners.T).T + t
        if (cam[:, 2] <= 0.3).any():
            continue
        proj = (K @ cam.T).T
        dst = (proj[:, :2] / proj[:, 2:3]).astype(np.float32)
        s = tex.shape[0] - 1
        Hm = cv2.getPerspectiveTransform(np.float32([[0,0],[s,0],[s,s],[0,s]]), dst)
        warped = cv2.warpPerspective(tex, Hm, (W, H))
        mask = cv2.warpPerspective(np.full(tex.shape[:2], 255, np.uint8), Hm, (W, H))
        img[mask > 0] = warped[mask > 0]
    return img

N = 8
work = tempfile.mkdtemp(prefix="sfm_")
imgdir = os.path.join(work, "images"); os.makedirs(imgdir)
for i in range(N):
    cv2.imwrite(os.path.join(imgdir, f"img{i:04d}.jpg"), render(i, N), [cv2.IMWRITE_JPEG_QUALITY, 92])
print(f"rendered {N} views of {W}x{H} to {imgdir}")

## OpenCV: features, matching, and epipolar geometry

In [ ]:
a = cv2.imread(os.path.join(imgdir, "img0000.jpg"), cv2.IMREAD_GRAYSCALE)
b = cv2.imread(os.path.join(imgdir, "img0001.jpg"), cv2.IMREAD_GRAYSCALE)

sift = cv2.SIFT_create()
k1, d1 = sift.detectAndCompute(a, None)
k2, d2 = sift.detectAndCompute(b, None)
print(f"SIFT keypoints: {len(k1)} and {len(k2)}")

good = [m for m, n in cv2.BFMatcher().knnMatch(d1, d2, k=2) if m.distance < 0.75 * n.distance]
print(f"ratio-test matches: {len(good)}")
assert len(good) > 50

p1 = np.float32([k1[m.queryIdx].pt for m in good])
p2 = np.float32([k2[m.trainIdx].pt for m in good])
E, mask = cv2.findEssentialMat(p1, p2, K, method=cv2.RANSAC)
n_in, R, t, _ = cv2.recoverPose(E, p1, p2, K, mask=mask.copy())
print(f"essential matrix inliers {int(mask.sum())}/{len(good)}, "
      f"recoverPose kept {n_in} points in front of both cameras")
assert n_in > 20

from skimage.feature import canny
edges = canny(a / 255.0, sigma=2)
print(f"scikit-image canny found {edges.sum()} edge pixels")

## pycolmap: the full sparse pipeline

Feature extraction, exhaustive matching and incremental mapping, all on CPU.
Dense MVS is not available on this image: it needs CUDA.

In [ ]:
import time
db = os.path.join(work, "database.db")
out = os.path.join(work, "sparse"); os.makedirs(out, exist_ok=True)

t0 = time.time()
pycolmap.extract_features(db, imgdir, camera_mode=pycolmap.CameraMode.SINGLE,
                          device=pycolmap.Device.cpu)
t_extract = time.time() - t0
print(f"extraction: {t_extract:.1f}s for {N} images = {t_extract/N:.2f}s/image")

t0 = time.time()
pycolmap.match_exhaustive(db, device=pycolmap.Device.cpu)
t_match = time.time() - t0
pairs = N * (N - 1) // 2
print(f"matching:   {t_match:.1f}s for {pairs} pairs = {t_match/pairs*1000:.0f}ms/pair")

t0 = time.time()
recs = pycolmap.incremental_mapping(db, imgdir, out)
print(f"mapping:    {time.time()-t0:.1f}s")

assert recs, "no reconstruction was produced"
rec = recs[min(recs)]  # incremental_mapping returns {index: Reconstruction}
print(f"\nregistered {rec.num_reg_images()}/{N} images, {rec.num_points3D()} 3D points")
cam = list(rec.cameras.values())[0]
print(f"recovered focal length {cam.params[0]:.1f} (true {F})")
assert rec.num_reg_images() >= N - 1
assert abs(cam.params[0] - F) / F < 0.15, "focal length is off by more than 15%"

print(f"\nat this rate, 300 images would need roughly:")
print(f"  extraction                 {t_extract/N*300/60:5.1f} min")
print(f"  exhaustive matching        {t_match/pairs*(300*299//2)/60:5.1f} min")
print(f"  sequential matching (10)   {t_match/pairs*(300*10)/60:5.1f} min")

In [ ]:
import shutil
shutil.rmtree(work, ignore_errors=True)
print("ALL CHECKS PASSED")